In [5]:
from keras.layers import Conv2D,MaxPooling2D,Dropout,Conv2DTranspose,Dense,Flatten,Input,concatenate
import tensorflow as tf
from keras import Model


In [7]:
def unet_model(input_size=(572, 572, 3)):
    inputs = Input(input_size)

    # Encoder
    c1 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(inputs)
    c1 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(p1)
    c2 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(p2)
    c3 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(p3)
    c4 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c4)
    p4 = MaxPooling2D((2, 2))(c4)

    # Bottleneck
    c5 = Conv2D(1024, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(p4)
    c5 = Conv2D(1024, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c5)

    # Decoder
    u6 = Conv2DTranspose(512, (2, 2), strides=(2, 2), padding='same')(c5)
    c4_cropped = tf.image.crop_to_bounding_box(c4, 4, 4, 56, 56)  # Cropping feature map
    u6 = concatenate([u6, c4_cropped])
    c6 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(u6)
    c6 = Conv2D(512, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c6)

    u7 = Conv2DTranspose(256, (2, 2), strides=(2, 2), padding='same')(c6)
    c3_cropped = tf.image.crop_to_bounding_box(c3, 16, 16, 104, 104)  # Cropping feature map
    u7 = concatenate([u7, c3_cropped])
    c7 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(u7)
    c7 = Conv2D(256, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c7)

    u8 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(c7)
    c2_cropped = tf.image.crop_to_bounding_box(c2, 40, 40, 200, 200)  # Cropping feature map
    u8 = concatenate([u8, c2_cropped])
    c8 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(u8)
    c8 = Conv2D(128, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c8)

    u9 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c8)
    c1_cropped = tf.image.crop_to_bounding_box(c1, 88, 88, 392, 392)  # Cropping feature map
    u9 = concatenate([u9, c1_cropped])
    c9 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(u9)
    c9 = Conv2D(64, (3, 3), activation='relu', kernel_initializer='he_normal', padding='valid')(c9)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = Model(inputs=[inputs], outputs=[outputs])
    return model
    

In [8]:
model = unet_model()
model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 572, 572, 3  0           []                               
                                )]                                                                
                                                                                                  
 conv2d (Conv2D)                (None, 570, 570, 64  1792        ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 conv2d_1 (Conv2D)              (None, 568, 568, 64  36928       ['conv2d[0][0]']                 
                                )                                                             

In [ ]:
history = model.fit(train_dataset,validation_data=val_dataset,batch_size=32)